<a href="https://colab.research.google.com/github/ARPITAKAR/100-days-of-machine-learning/blob/main/Garbage_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [2]:
!kaggle datasets download -d sumn2u/garbage-classification-v2

Dataset URL: https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2
License(s): MIT
 94% 703M/744M [00:04<00:01, 42.9MB/s]
100% 744M/744M [00:04<00:00, 193MB/s] 


In [3]:
import zipfile
zip_ref= zipfile.ZipFile('/content/garbage-classification-v2.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [4]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Dropout

In [5]:
import os
from PIL import Image
from collections import defaultdict

dataset_dir = "/content/garbage-dataset"

# Dictionary: folder → {total files, bad files}
report = defaultdict(lambda: {"total": 0, "bad": []})

for root, _, files in os.walk(dataset_dir):
    for f in files:
        filepath = os.path.join(root, f)
        folder = os.path.basename(root)  # class folder name
        report[folder]["total"] += 1

        try:
            Image.open(filepath).verify()
        except Exception:
            report[folder]["bad"].append(filepath)

# Print report
for folder, stats in report.items():
    print(f"\nFolder: {folder}")
    print(f"  Total files: {stats['total']}")
    print(f"  Corrupted/unsupported: {len(stats['bad'])}")
    for bad in stats['bad']:
        print("   -", bad)





Folder: metal
  Total files: 1020
  Corrupted/unsupported: 0

Folder: glass
  Total files: 3061
  Corrupted/unsupported: 0

Folder: cardboard
  Total files: 1825
  Corrupted/unsupported: 0

Folder: trash
  Total files: 947
  Corrupted/unsupported: 0

Folder: battery
  Total files: 944
  Corrupted/unsupported: 0

Folder: shoes
  Total files: 1977
  Corrupted/unsupported: 0

Folder: paper
  Total files: 1680
  Corrupted/unsupported: 0

Folder: clothes
  Total files: 5327
  Corrupted/unsupported: 0

Folder: plastic
  Total files: 1984
  Corrupted/unsupported: 0

Folder: biological
  Total files: 997
  Corrupted/unsupported: 0


In [6]:
dataset_dir = "/content/garbage-dataset"

# Supported extensions (lowercase)
valid_exts = {".jpg", ".jpeg", ".png", ".gif", ".bmp"}

report = defaultdict(lambda: {"total": 0, "unsupported": []})

for root, _, files in os.walk(dataset_dir):
    for f in files:
        filepath = os.path.join(root, f)
        folder = os.path.basename(root)
        report[folder]["total"] += 1

        # check extension
        ext = os.path.splitext(f)[1].lower()
        if ext not in valid_exts:
            report[folder]["unsupported"].append(filepath)

# Print report
for folder, stats in report.items():
    print(f"\nFolder: {folder}")
    print(f"  Total files: {stats['total']}")
    print(f"  Unsupported format: {len(stats['unsupported'])}")
    for bad in stats['unsupported']:
        print("   -", bad)



Folder: metal
  Total files: 1020
  Unsupported format: 0

Folder: glass
  Total files: 3061
  Unsupported format: 0

Folder: cardboard
  Total files: 1825
  Unsupported format: 0

Folder: trash
  Total files: 947
  Unsupported format: 0

Folder: battery
  Total files: 944
  Unsupported format: 0

Folder: shoes
  Total files: 1977
  Unsupported format: 0

Folder: paper
  Total files: 1680
  Unsupported format: 0

Folder: clothes
  Total files: 5327
  Unsupported format: 0

Folder: plastic
  Total files: 1984
  Unsupported format: 0

Folder: biological
  Total files: 997
  Unsupported format: 0


In [7]:
import os
import imghdr
from collections import defaultdict

dataset_dir = "/content/garbage-dataset"

report = defaultdict(lambda: {"total": 0, "bad_format": []})

for root, _, files in os.walk(dataset_dir):
    for f in files:
        filepath = os.path.join(root, f)
        folder = os.path.basename(root)
        report[folder]["total"] += 1

        # Check actual file signature
        kind = imghdr.what(filepath)
        if kind not in ["jpeg", "png", "gif", "bmp"]:
            report[folder]["bad_format"].append((filepath, kind))

# Print summary
for folder, stats in report.items():
    print(f"\nFolder: {folder}")
    print(f"  Total files: {stats['total']}")
    print(f"  Wrongly formatted: {len(stats['bad_format'])}")
    for bad_file, detected in stats["bad_format"]:
        print(f"   - {bad_file} (detected: {detected})")


/tmp/ipython-input-3696965723.py:2: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr



Folder: metal
  Total files: 1020
  Wrongly formatted: 0

Folder: glass
  Total files: 3061
  Wrongly formatted: 0

Folder: cardboard
  Total files: 1825
  Wrongly formatted: 2
   - /content/garbage-dataset/cardboard/cardboard_2088.jpg (detected: None)
   - /content/garbage-dataset/cardboard/cardboard_313.jpg (detected: None)

Folder: trash
  Total files: 947
  Wrongly formatted: 0

Folder: battery
  Total files: 944
  Wrongly formatted: 0

Folder: shoes
  Total files: 1977
  Wrongly formatted: 0

Folder: paper
  Total files: 1680
  Wrongly formatted: 6
   - /content/garbage-dataset/paper/paper_2784.jpg (detected: webp)
   - /content/garbage-dataset/paper/paper_1433.jpg (detected: None)
   - /content/garbage-dataset/paper/paper_1678.jpg (detected: webp)
   - /content/garbage-dataset/paper/paper_3119.jpg (detected: webp)
   - /content/garbage-dataset/paper/paper_2779.jpg (detected: None)
   - /content/garbage-dataset/paper/paper_2184.jpg (detected: None)

Folder: clothes
  Total files:

In [8]:
bad_files = [
    "/content/garbage-dataset/cardboard/cardboard_2088.jpg",
    "/content/garbage-dataset/cardboard/cardboard_313.jpg",
    "/content/garbage-dataset/paper/paper_2784.jpg",
    "/content/garbage-dataset/paper/paper_1433.jpg",
    "/content/garbage-dataset/paper/paper_1678.jpg",
    "/content/garbage-dataset/paper/paper_3119.jpg",
    "/content/garbage-dataset/paper/paper_2779.jpg",
    "/content/garbage-dataset/paper/paper_2184.jpg",
    "/content/garbage-dataset/plastic/plastic_160.jpg",
    "/content/garbage-dataset/plastic/plastic_603.jpg",
    "/content/garbage-dataset/plastic/plastic_613.jpg",
    "/content/garbage-dataset/plastic/plastic_2038.jpg",
]

for f in bad_files:
    try:
        os.remove(f)
        print("Deleted:", f)
    except Exception as e:
        print("Could not delete:", f, "->", e)


Deleted: /content/garbage-dataset/cardboard/cardboard_2088.jpg
Deleted: /content/garbage-dataset/cardboard/cardboard_313.jpg
Deleted: /content/garbage-dataset/paper/paper_2784.jpg
Deleted: /content/garbage-dataset/paper/paper_1433.jpg
Deleted: /content/garbage-dataset/paper/paper_1678.jpg
Deleted: /content/garbage-dataset/paper/paper_3119.jpg
Deleted: /content/garbage-dataset/paper/paper_2779.jpg
Deleted: /content/garbage-dataset/paper/paper_2184.jpg
Deleted: /content/garbage-dataset/plastic/plastic_160.jpg
Deleted: /content/garbage-dataset/plastic/plastic_603.jpg
Deleted: /content/garbage-dataset/plastic/plastic_613.jpg
Deleted: /content/garbage-dataset/plastic/plastic_2038.jpg


In [9]:
# generators
train_ds=keras.utils.image_dataset_from_directory(
    directory='/content/garbage-dataset',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256),
    validation_split=0.2,  # Using 20% of the data for validation
    subset='training',
    seed=123 # Setting a seed for reproducibility
)

Found 19750 files belonging to 10 classes.
Using 15800 files for training.


In [10]:
validation_ds = keras.utils.image_dataset_from_directory(
    directory='/content/garbage-dataset',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256),
    validation_split=0.2,  # Using 20% of the data for validation
    subset='validation',
    seed=123 # Setting a seed for reproducibility
)

Found 19750 files belonging to 10 classes.
Using 3950 files for validation.


In [11]:
# Noramlize
def process(image,label):
  image= tf.cast(image/255. ,tf.float32)
  return image,label

train_ds=train_ds.map(process)
validation_ds=validation_ds.map(process)

In [12]:
#create a CNN Model
model = Sequential()
model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(10,activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 125, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,778 (56.64 MB)

 Trainable params: 14,848,330 (56.64 MB)

 Non-trainable params: 448 (1.75 KB)

In [14]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
history=model.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 69s 112ms/step - accuracy: 0.2667 - loss: 5.2880 - val_accuracy: 0.2658 - val_loss: 2.0648
Epoch 2/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 65s 99ms/step - accuracy: 0.3387 - loss: 1.9949 - val_accuracy: 0.3405 - val_loss: 1.9012
Epoch 3/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 79s 93ms/step - accuracy: 0.3702 - loss: 1.8379 - val_accuracy: 0.3777 - val_loss: 1.9362
Epoch 4/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 49s 98ms/step - accuracy: 0.3861 - loss: 1.7681 - val_accuracy: 0.3813 - val_loss: 1.8275
Epoch 5/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 49s 98ms/step - accuracy: 0.4140 - loss: 1.6843 - val_accuracy: 0.4225 - val_loss: 1.6679
Epoch 6/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 46s 93ms/step - accuracy: 0.4457 - loss: 1.5729 - val_accuracy: 0.4468 - val_loss: 1.5970
Epoch 7/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 45s 91ms/step - accuracy: 0.4802 - loss: 1.4864 - val_accuracy: 0.4904 - val_loss: 1.5284
Epoch 8/10
494/494 ━━━━━━━━━━━━━━━━━━━━ 87s 102ms/step - accuracy: 0.5015 - loss: 1.4092 